In [ ]:
# ============================================================
# CALCULATED QUANTITIES:
# 1. Global polar order parameter, OP
# 2. Polar order parameter inside the non-noisy region, OP_in
# 3. Particle number inside the non-noisy region, N_in
# 4. Particle density inside the non-noisy region, rho_in
#
# All quantities are averaged over the last 50 timesteps and
# then averaged over independent simulation runs.
# ============================================================

import os
import glob
import re
import numpy as np
import pandas as pd


# ============================================================
# CONFIGURATION
# ============================================================

INPUT_FOLDER = "input_folder"
OUTPUT_FOLDER = "analysis_results"
N_TIMESTEPS = 50
CENTER_X = 10.0
CENTER_Y = 10.0


# ============================================================
# FUNCTIONS
# ============================================================

def order_parameter(directions):
    if len(directions) == 0:
        return np.nan
    return np.linalg.norm(np.mean(np.column_stack((np.cos(directions), np.sin(directions))), axis=0))


def extract_parameters(filepath):
    fac_match = re.search(r"fac([\d.]+)", filepath)
    srad_match = re.search(r"srad([\d.]+)", filepath)
    if fac_match is None or srad_match is None:
        return None, None
    return float(fac_match.group(1)), float(srad_match.group(1))


def read_data(filepath):
    df = pd.read_csv(filepath, sep=r"\s+", engine="python")
    if df["direction"].max() > 2 * np.pi:
        df["direction"] = np.deg2rad(df["direction"])
    return df


def calculate_observables(filepath, n_timesteps=N_TIMESTEPS):
    df = read_data(filepath)
    fac, srad = extract_parameters(filepath)
    times = np.sort(df["time"].unique())[-n_timesteps:]
    op_global, op_inside, n_inside, rho_inside = [], [], [], []
    area = np.pi * srad**2

    for t in times:
        data = df[df["time"] == t]
        global_directions = data["direction"].values
        distances = np.sqrt((data["x"] - CENTER_X)**2 + (data["y"] - CENTER_Y)**2)
        inside = data[distances <= srad]
        inside_directions = inside["direction"].values
        n_in = len(inside)

        op_global.append(order_parameter(global_directions))
        op_inside.append(order_parameter(inside_directions))
        n_inside.append(n_in)
        rho_inside.append(n_in / area)

    return np.nanmean(op_global), np.nanmean(op_inside), np.mean(n_inside), np.mean(rho_inside)


def save_results(results):
    for srad, data in results.items():
        output_dir = os.path.join(OUTPUT_FOLDER, f"srad{srad}")
        os.makedirs(output_dir, exist_ok=True)
        df = pd.DataFrame(data).sort_values("fac")
        df.to_csv(os.path.join(output_dir, "order_density_vs_fac.csv"), index=False)


# ============================================================
# MAIN
# ============================================================

def main():
    os.makedirs(OUTPUT_FOLDER, exist_ok=True)
    files = sorted(glob.glob(os.path.join(INPUT_FOLDER, "fac*", "srad*", "run_*.txt")))

    if not files:
        raise FileNotFoundError(f"No simulation files found in {INPUT_FOLDER}")

    groups = {}
    for filepath in files:
        fac, srad = extract_parameters(filepath)
        if fac is not None and srad is not None:
            groups.setdefault((fac, srad), []).append(filepath)

    results = {}

    for (fac, srad), file_list in sorted(groups.items()):
        observables = [calculate_observables(filepath) for filepath in file_list]
        observables = np.asarray(observables)

        op_global = observables[:, 0]
        op_inside = observables[:, 1]
        n_inside = observables[:, 2]
        rho_inside = observables[:, 3]

        results.setdefault(srad, []).append({"fac": fac, "OP": np.nanmean(op_global), "OP_in": np.nanmean(op_inside), "N_in": np.nanmean(n_inside), "rho_in": np.nanmean(rho_inside), "OP_std": np.nanstd(op_global), "OP_in_std": np.nanstd(op_inside), "N_in_std": np.nanstd(n_inside), "rho_in_std": np.nanstd(rho_inside)})

    save_results(results)


if __name__ == "__main__":
    main()